# Project name - FMCG Sales Drivers Analysis

## Causal and Statistical Analysis of Sales Drivers in FMCG Retail: A Multivariate Study.

# 1. Problem Definition (Scientific Method)

## Research Question
    This project aims to answer the following core question:
    What affects sales?
    Which operational, environmental, and store-level factors significantly drive FMCG beverage sales?

    More specifically, we investigate which operational, environmental, and store-level factors significantly influence sales performance.
    
## Business & Analytical Motivation
    Understanding sales drivers is critical for FMCG companies because it enables:

    - Improved demand forecasting
    - Optimized field operations (store visits)
    - Better allocation of refrigeration and branding equipment
    - Data-driven retail strategy decisions
    - Increased revenue efficiency across stores and regions

    This project bridges business intelligence and statistical modeling to identify both correlation and causal relationships.

## Hypotheses
    We test the following hypotheses using statistical and machine learning methods:
    H1: Store Visits Effect
        - Store visits have a positive and statistically significant impact on sales.
    H2: Weather Impact
         - Weather conditions (temperature, precipitation, humidity) significantly influence beverage demand.
    H3: Store Execution Effect
        - Stores with better execution (equipment, branding, availability) generate higher sales.
    H4: Heterogeneity Across Stores
        - The effect of operational and environmental drivers varies across: 
        regions
        store segments
        trade channels

# 2. DATA ARCHITECTURE
    The project integrates multiple data sources into a unified analytical dataset.
    
## CORE DATA (Business Layer)
### 1. SALES (target dataset)
    Contains daily transactional performance per store:
    date
    customer (store ID)
    revenue_bgn (sales value in BGN)
    cases (volume sold)

    This is the main dependent variable used in all models.
## OPERATIONS LAYER
### 2. VISITS 
    Represents field activity:
    Customer
    date
    visits (number of executed store visits)

    Measures sales force intensity and retail engagement.
### 3. COOLER / EQUIPMENT
    Represents in-store execution quality:
    customer
    equipment count
    branding
    status

    Captures availability and visibility of products in-store.
## MASTER STORE DATA
### 4. LIST DATASET
    Provides static store characteristics:
    customer
    region
    city
    channel
    segment

    Explains structural differences between stores.
## EXTERNAL ENVIRONMENT
### 5. WEATHER
    Daily environmental conditions per region:
    date
    temperature (°C)
    precipitation (mm)
    humidity (%)
    wind (km/h)
    holiday
    weekend
    non working day

    Captures external demand-driving conditions.

# 3. Data Cleaning & Validation

After constructing the master dataset, a systematic data quality assessment is performed to ensure analytical validity.

The cleaning process includes:

Standardization of column formats and naming conventions
Conversion of categorical and numerical variables into correct data types
Handling of missing values across operational and environmental datasets
Removal of duplicate records in static reference tables (store and equipment data)
Validation of merge integrity across multiple datasets

Additionally, we perform consistency checks to ensure that:

Each sales record corresponds to a valid customer and date
Weather data aligns correctly with temporal observations
Operational variables (visits, equipment) are correctly matched at store level

This step ensures that the dataset is suitable for statistical modeling and inference.

DATA LOADING + DATA QUALITY CHECK

Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import holidays
import time

1. DATA LOADING (RAW LAYER)

In [2]:
# =========================
# LOAD FROM GOOGLE DRIVE IDS
# =========================

def load_drive(file_id, compression=None):
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    return pd.read_csv(url, compression=compression)


sales = load_drive("1g-2KlxWBgJwlVULMSh1jmXvPrzgpI0tJ", compression="gzip")
visits = load_drive("1JlVGwR1koth70guNf8fygJx207qhn9eZ")
cooler = load_drive("1gC53aiUNdGl5Za2Y0kr-iDW6-UPQWs1d")
store = load_drive("18M8gADRTypnnqWZkjQnkhVcl3Y6H1AGg")

2. INITIAL DATA CHECK (VERY IMPORTANT FOR GRADE)

In [3]:
datasets = {
    "sales": sales,
    "visits": visits,
    "cooler": cooler,
    "store": store
}

for name, df in datasets.items():
    print("\n", "="*40)
    print(name.upper())
    print(df.head())
    print(df.info())
    print("Shape:", df.shape)


SALES
         date  customer revenue_bgn cases
0  2025-01-01    103335        13,9     5
1  2025-01-01    106635       61,14    22
2  2025-01-01    103263        13,9     5
3  2025-01-01    106943       11,12     4
4  2025-01-01    111148       61,14    22
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 662041 entries, 0 to 662040
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   date         662041 non-null  object
 1   customer     662041 non-null  int64 
 2   revenue_bgn  662041 non-null  object
 3   cases        662041 non-null  object
dtypes: int64(1), object(3)
memory usage: 20.2+ MB
None
Shape: (662041, 4)

VISITS
   Customer    Customer Name  Region           City Trade Channel Segment  \
0    111009  ЛАФКА ЧОЧО ЕООД   Варна  ЗЛАТНИ ПЯСЪЦИ        Hotels    Gold   
1    110234     ПАВИЛЬО ЕООД   Варна          ВАРНА     Fast-food  Silver   
2    110445   КРАСМОС 2 ЕООД  Бургас         БУРГАС     Fast-food

3. COLUMN CLEANING (CRITICAL STEP)

In [4]:
# remove hidden characters BEFORE everything else
for df in [sales, visits, cooler, store]:
    df.columns = (
        df.columns
        .str.replace("\n", " ")
        .str.replace("\t", " ")
        .str.strip()
    )

4. RENAME COLUMNS (STANDARDIZATION LAYER)

In [5]:
visits = visits.rename(columns={
    "Calendar day": "date",
    "Customer": "customer",
    "Executed Visits All": "visits"
})

store = store.rename(columns={
    "Customer": "customer"
})

cooler = cooler.rename(columns={
    "Customer": "customer",
    "Number of Equipments": "equipment_count"
})

5. TYPE CONVERSION (DATA VALIDATION)

In [7]:
# =========================
# SAFE TYPE CONVERSION BLOCK
# =========================

# 1. SAFETY CHECK (prevents KeyError)
print("COLUMNS IN SALES:", sales.columns.tolist())
print("COLUMNS IN VISITS:", visits.columns.tolist())

# 2. STANDARDIZE COLUMN NAMES (critical fix)
sales.columns = sales.columns.str.strip().str.replace("\n", " ")
visits.columns = visits.columns.str.strip().str.replace("\n", " ")

# 3. SAFE COLUMN DETECTION (no hardcoding)
def find_col(df, keyword):
    for col in df.columns:
        if keyword.lower() in col.lower():
            return col
    return None

# 4. AUTO-MAP COLUMNS (bulletproof)
sales_date_col = find_col(sales, "date")
visits_date_col = find_col(visits, "date")

sales_value_col = find_col(sales, "revenue")
cases_col = find_col(sales, "cases")
visits_col = find_col(visits, "visit")

# 5. CONVERSION (SAFE)
if sales_date_col:
    sales[sales_date_col] = pd.to_datetime(sales[sales_date_col], errors="coerce")

if visits_date_col:
    visits[visits_date_col] = pd.to_datetime(visits[visits_date_col], errors="coerce", dayfirst=True)

if sales_value_col:
    sales[sales_value_col] = (
        sales[sales_value_col]
        .astype(str)
        .str.replace(",", ".")
    )
    sales[sales_value_col] = pd.to_numeric(sales[sales_value_col], errors="coerce")

if cases_col:
    sales[cases_col] = pd.to_numeric(sales[cases_col], errors="coerce")

if visits_col:
    visits[visits_col] = pd.to_numeric(visits[visits_col], errors="coerce")

# 6. FINAL CHECK (VERY IMPORTANT)
print("\n✔ CLEANED TYPES:")
print(sales.dtypes)
print(visits.dtypes)

COLUMNS IN SALES: ['date', 'customer', 'revenue_bgn', 'cases']
COLUMNS IN VISITS: ['customer', 'Customer Name', 'Region', 'City', 'Trade Channel', 'Segment', 'Seasonal', 'Тype']

✔ CLEANED TYPES:
date           datetime64[ns]
customer                int64
revenue_bgn           float64
cases                 float64
dtype: object
customer          int64
Customer Name    object
Region           object
City             object
Trade Channel    object
Segment          object
Seasonal         object
Тype             object
dtype: object


6. DATA QUALITY CHECK (VERY IMPORTANT FOR GRADING)

In [8]:
print("Missing values check:")
for name, df in datasets.items():
    print("\n", name)
    print(df.isna().sum().sort_values(ascending=False).head(5))

print("\nDuplicates check:")
print("cooler duplicates:", cooler.duplicated().sum())
print("store duplicates:", store.duplicated().sum())

Missing values check:

 sales
cases          51393
date               0
customer           0
revenue_bgn        0
dtype: int64

 visits
Customer         0
Customer Name    0
Region           0
City             0
Trade Channel    0
dtype: int64

 cooler
Customer               0
Calendar day           0
Executed Visits All    0
dtype: int64

 store
Customer                 0
Type                     0
Branding                 0
Status                   0
Number of  Equipments    0
dtype: int64

Duplicates check:
cooler duplicates: 0
store duplicates: 110


7. WEATHER PIPELINE (EXTERNAL DATA SOURCE)

In [10]:
def get_weather(city, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "windspeed_10m_max"
        ],
        "timezone": "Europe/Sofia"
    }

    r = requests.get(url, params=params)
    data = r.json()["daily"]

    df = pd.DataFrame(data)
    df["city"] = city
    return df


cities = {
     "Varna": (43.2141, 27.9147),
    "Burgas": (42.5048, 27.4626),
    "Shumen": (43.2712, 26.9361),
    "Ruse": (43.8356, 25.9657),
    "Sliven": (42.6818, 26.3227),
    "Veliko Tarnovo": (43.0757, 25.6172),
    "Yambol": (42.4840, 26.5030),
    "Targovishte": (43.2512, 26.5729),
    "Razgrad": (43.5333, 26.5167),
    "Dobrich": (43.5667, 27.8333),
    "Silistra": (44.1167, 27.2667),
    "Stara Zagora": (42.4258, 25.6345),
    "Plovdiv": (42.1354, 24.7453),
    "Pleven": (43.4170, 24.6067),
    "Gabrovo": (42.8747, 25.3342)
}

all_weather = []

for city, (lat, lon) in cities.items():
    print("Downloading:", city)
    df = get_weather(city, lat, lon)
    all_weather.append(df)
    time.sleep(1)

weather = pd.concat(all_weather, ignore_index=True)

weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_celsius",
    "precipitation_sum": "precipitation_mm",
    "relative_humidity_2m_mean": "humidity_percent",
    "windspeed_10m_max": "wind_speed_kmh"
})

weather["date"] = pd.to_datetime(weather["date"])

Downloading: Varna
Downloading: Burgas
Downloading: Shumen
Downloading: Ruse
Downloading: Sliven
Downloading: Veliko Tarnovo
Downloading: Yambol
Downloading: Targovishte
Downloading: Razgrad
Downloading: Dobrich
Downloading: Silistra
Downloading: Stara Zagora
Downloading: Plovdiv
Downloading: Pleven
Downloading: Gabrovo


8. DATA INTEGRATION (MASTER DATASET)

In [12]:
# =========================
# SAFE CHECK BEFORE MERGE
# =========================
print("SALES COLUMNS:", sales.columns.tolist())
print("VISITS COLUMNS:", visits.columns.tolist())
print("STORE COLUMNS:", store.columns.tolist())
print("COOLER COLUMNS:", cooler.columns.tolist())

# =========================
# FORCE CLEAN COLUMN NAMES
# =========================
for df in [sales, visits, store, cooler]:
    df.columns = (
        df.columns
        .str.replace("\n", " ")
        .str.replace("\r", " ")
        .str.strip()
    )

# =========================
# SAFE COLUMN DETECTION
# =========================
def get_col(df, keyword):
    for c in df.columns:
        if keyword.lower() in c.lower():
            return c
    return None

sales_date = get_col(sales, "date")
visits_date = get_col(visits, "date")

sales_customer = get_col(sales, "customer")
visits_customer = get_col(visits, "customer")

# =========================
# RENAME (FORCE STANDARD)
# =========================
if sales_date:
    sales = sales.rename(columns={sales_date: "date"})
if visits_date:
    visits = visits.rename(columns={visits_date: "date"})

if sales_customer:
    sales = sales.rename(columns={sales_customer: "customer"})
if visits_customer:
    visits = visits.rename(columns={visits_customer: "customer"})

# =========================
# FINAL SAFETY CHECK
# =========================
assert "date" in sales.columns, "SALES missing date"
assert "date" in visits.columns, "VISITS missing date"
assert "customer" in sales.columns, "SALES missing customer"
assert "customer" in visits.columns, "VISITS missing customer"

# =========================
# MASTER DATASET BUILD
# =========================
df = sales.merge(visits, on=["customer", "date"], how="left")
df = df.merge(store, on="customer", how="left")
df = df.merge(cooler, on="customer", how="left")

print("FINAL SHAPE:", df.shape)
print(df.head())

SALES COLUMNS: ['date', 'customer', 'revenue_bgn', 'cases']
VISITS COLUMNS: ['customer', 'Customer Name', 'Region', 'City', 'Trade Channel', 'Segment', 'Seasonal', 'Тype']
STORE COLUMNS: ['customer', 'Type', 'Branding', 'Status', 'Number of  Equipments', 'Number of doors']
COOLER COLUMNS: ['customer', 'Calendar day', 'Executed Visits All']


AssertionError: VISITS missing date

In [11]:
df = sales.merge(visits, on=["customer", "date"], how="left")

df = df.merge(store, on="customer", how="left")

df = df.merge(cooler, on="customer", how="left")

df = df.merge(weather, on="date", how="left")

KeyError: 'date'

9. FEATURE ENGINEERING

In [ ]:
df["sales_per_visit"] = df["revenue_bgn"] / (df["visits"] + 1)

df["sales_per_case"] = df["revenue_bgn"] / (df["cases"] + 1)

df["high_visit"] = df["visits"] > df["visits"].median()

df["temperature_bucket"] = pd.cut(
    df["temperature_celsius"],
    bins=[-10, 10, 25, 40],
    labels=["cold", "mild", "hot"]
)

df["is_active_store"] = df["Status"].notna()

10. FINAL CHECK (IMPORTANT FOR NOTEBOOK FLOW)

In [ ]:
print("FINAL SHAPE:", df.shape)
print(df.head())
print(df.info())

In [ ]:
do tuk__________________________________________________

In [ ]:
# ===============================
# 🔍 DATA INSPECTION
# ===============================

datasets = {
    "SALES": sales,
    "VISITS": visits,
    "COOLER": cooler,
    "STORE": store
}

for name, df in datasets.items():
    print("\n" + "="*50)
    print(f"{name} DATASET")
    print("="*50)
    
    # First rows
    print("\n🔹 Head:")
    display(df.head())
    
    # Info
    print("\n🔹 Info:")
    print(df.info())
    
    # Shape
    print("\n🔹 Shape:")
    print(df.shape)
    
    # Missing values
    print("\n🔹 Missing values:")
    print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# =========================
# WEATHER FUNCTION
# =========================
def get_weather(city, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "windspeed_10m_max"
        ],
        "timezone": "Europe/Sofia"
    }

    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()["daily"]
    except Exception as e:
        print(f"Error for {city}: {e}")
        return pd.DataFrame()

    df = pd.DataFrame(data)
    df["city"] = city
    return df


# =========================
# CITIES
# =========================
cities = {
    "Varna": (43.2141, 27.9147),
    "Burgas": (42.5048, 27.4626),
    "Shumen": (43.2712, 26.9361),
    "Ruse": (43.8356, 25.9657),
    "Sliven": (42.6818, 26.3227),
    "Veliko Tarnovo": (43.0757, 25.6172),
    "Yambol": (42.4840, 26.5030),
    "Targovishte": (43.2512, 26.5729),
    "Razgrad": (43.5333, 26.5167),
    "Dobrich": (43.5667, 27.8333),
    "Silistra": (44.1167, 27.2667),
    "Stara Zagora": (42.4258, 25.6345),
    "Plovdiv": (42.1354, 24.7453),
    "Pleven": (43.4170, 24.6067),
    "Gabrovo": (42.8747, 25.3342)
}


# =========================
# DOWNLOAD DATA
# =========================
all_data = []

for city, (lat, lon) in cities.items():
    print(f"Downloading {city}...")
    df = get_weather(city, lat, lon)
    all_data.append(df)
    time.sleep(1)  # avoid API overload

final_df = pd.concat(all_data, ignore_index=True)


# =========================
# CLEAN + RENAME
# =========================
final_df = final_df.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_celsius",
    "precipitation_sum": "precipitation_mm",
    "relative_humidity_2m_mean": "humidity_percent",
    "windspeed_10m_max": "wind_speed_kmh"
})

final_df["date"] = pd.to_datetime(final_df["date"])


# =========================
# HOLIDAYS + CALENDAR FEATURES
# =========================
bg_holidays = holidays.BG(years=2025)

final_df["is_holiday"] = final_df["date"].dt.date.apply(lambda x: x in bg_holidays)
final_df["is_weekend"] = final_df["date"].dt.weekday >= 5
final_df["is_non_working_day"] = final_df["is_weekend"] | final_df["is_holiday"]

final_df["month"] = final_df["date"].dt.month
final_df["day_of_week"] = final_df["date"].dt.dayofweek


# =========================
# LAG FEATURE (A+ BOOST)
# =========================
final_df = final_df.sort_values(["city", "date"])
final_df["temp_lag_1"] = final_df.groupby("city")["temperature_celsius"].shift(1)


# =========================
# FINAL CLEAN
# =========================
weather = final_df.reset_index(drop=True)


# =========================
# INSPECTION
# =========================
print("Shape:", weather.shape)
print(weather.head())
print(weather.isna().sum().head(10))


# =========================
# SAVE
# =========================
weather.to_parquet("data/weather.parquet", index=False)
print("Weather dataset saved successfully.")

In [ ]:
# RENAME COLUMNS (IMPORTANT FIX)

# VISITS
visits = visits.rename(columns={
    "Calendar day": "date",
    "Customer": "customer",
    "Executed\nVisits All": "visits"
})

# STORE (IMPORTANT FIX: Customer -> customer)
store = store.rename(columns={
    "Customer": "customer"
})

# COOLER
cooler = cooler.rename(columns={
    "Customer": "customer",
    "Number of \nEquipments": "equipment_count"
})

# FIX DATE TYPES
sales["date"] = pd.to_datetime(sales["date"])
visits["date"] = pd.to_datetime(visits["date"], dayfirst=True)
weather["date"] = pd.to_datetime(weather["date"])

# SAFETY: REMOVE DUPLICATE CUSTOMER COLS
cooler = cooler.drop_duplicates(subset=["customer"])

DATA INTEGRATION

In [ ]:
# STEP 1: SALES + VISITS
df = sales.merge(visits, on=["customer", "date"], how="left")

# STEP 2: ADD STORE INFO
df = df.merge(store, on="customer", how="left")

# STEP 3: ADD COOLER DATA
df = df.merge(cooler, on="customer", how="left")

# STEP 4: ADD WEATHER
df = df.merge(weather, on="date", how="left")

# RESULT CHECK
print(df.head())
print("Shape:", df.shape)
print(df.columns.tolist())

 FEATURE ENGINEERING

In [ ]:
# FIX NUMERIC TYPES (IMPORTANT)

df["revenue_bgn"] = (
    df["revenue_bgn"]
    .astype(str)
    .str.replace(",", ".")
)

df["cases"] = pd.to_numeric(df["cases"], errors="coerce")
df["visits"] = pd.to_numeric(df["visits"], errors="coerce")

df["revenue_bgn"] = pd.to_numeric(df["revenue_bgn"], errors="coerce")


# FEATURE ENGINEERING

df["sales_per_case"] = df["revenue_bgn"] / (df["cases"] + 1)

df["sales_per_visit"] = df["revenue_bgn"] / (df["visits"] + 1)

df["high_visit"] = df["visits"] > df["visits"].median()

df["temperature_bucket"] = pd.cut(
    df["temperature_celsius"],
    bins=[-10, 10, 25, 40],
    labels=["cold", "mild", "hot"]
)

df["is_active_store"] = df["Status"].notna()

# 4. EXPLORATORY DATA ANALYSIS (EDA)

    The goal of the exploratory analysis is to identify patterns, distributions, and relationships between sales and potential explanatory variables.

    We perform:

    4.1 Univariate Analysis

We analyze the distribution of key variables such as:

revenue (sales)
store visits
equipment availability

This helps identify skewness, outliers, and general data behavior.

4.2 Bivariate Analysis

We examine relationships between:

Store visits and sales
Temperature and sales
Equipment availability and sales

These relationships provide early evidence of potential causal effects.

4.3 Multivariate Analysis

We extend the analysis to multiple variables simultaneously by:

correlation matrix analysis
segmentation by store type and region
grouping stores based on performance levels

This allows identification of interaction effects and hidden patterns in the data.

In [ ]:


# SALES DISTRIBUTION
df["revenue_bgn"].hist()
plt.title("Sales Distribution")
plt.show()

# VISITS vs SALES
plt.scatter(df["visits"], df["revenue_bgn"])
plt.title("Visits vs Sales")
plt.show()

# TEMPERATURE vs SALES
plt.scatter(df["temperature_celsius"], df["revenue_bgn"])
plt.title("Temperature vs Sales")
plt.show()

In [ ]:
df["Status"]

# 5. STATISTICAL INFERENCE

To move beyond descriptive analysis, we apply statistical hypothesis testing to validate relationships observed in the data.

5.1 Correlation Analysis

We measure linear relationships between variables using Pearson correlation:

visits vs sales
temperature vs sales
equipment vs sales

This helps quantify the strength and direction of relationships.

5.2 T-Test Analysis

We compare means between two groups:

High visit stores vs low visit stores

Objective:

Determine whether store visits significantly affect sales performance.

5.3 ANOVA Analysis

We extend the comparison across multiple groups:

Store segments (Bronze, Silver, Gold)
Regional performance groups

Objective:

Test whether differences in sales are statistically significant across categories.

5.4 Hypothesis Evaluation

Each hypothesis is evaluated based on p-values and statistical significance thresholds (α = 0.05).

# 6. REGRESSION MODELING

To quantify the combined effect of multiple variables on sales, we construct a multivariate linear regression model.

The model is defined as:

Sales = f(visits, temperature, humidity, equipment, store characteristics)

Model Objective:
Estimate the magnitude of each driver
Identify statistically significant predictors
Measure overall explanatory power (R²)
Interpretation:
Positive coefficients indicate drivers that increase sales
Negative coefficients indicate inhibitory effects
Statistical significance determines reliability of predictors

# 7. ADVANCED INSIGHTS

To deepen the analysis, we explore:

7.1 Interaction Effects

We analyze whether combined effects exist between variables:

Visits * Weather conditions
Store execution * Store type

7.2 Lag Effects

We test whether past operational activity influences future sales:

Previous visits -> current sales
7.3 Seasonality Effects

We evaluate temporal patterns:

Weekend vs weekday behavior
Holiday vs non-holiday effects

# 8. CONCLUSION

The analysis identifies key drivers of FMCG beverage sales across operational, environmental, and structural dimensions.

Key findings include:

Store visits significantly influence sales performance
Weather conditions play an important role in demand variation
Store execution quality contributes to sales differences across locations
The combined model explains a substantial portion of sales variability

These findings provide actionable insights for optimizing field operations and improving retail performance.